In [ ]:
from dinov2.data.datasets.s2_csv import S2CsvDataset
from torchvision import transforms
from torch.utils.data import DataLoader

# 先算一次均值方差（可选）
# from dinov2.utils.data import compute_dataset_stats
# tmp_ds = S2CsvDataset(csv_path="data_csv/96_3_train.csv", scale_to_unit=True, pad_to_multiple=14, normalize_stats=None)
# compute_dataset_stats(tmp_ds, subset=0.1)  # 输出 mean/std，填到 normalize_stats

train_tf = transforms.Compose([
    # 这里可以插入你需要的随机裁剪/翻转
])

ds_train = S2CsvDataset(
    csv_path="data_csv/96_3_train.csv",
    ds_cfg_name="s2_12band",
    normalize_stats=None,   # 如果已有 mean/std，填为 (mean, std)
    scale_to_unit=True,     # uint16 -> [0,1]
    pad_to_multiple=14,     # 96 -> 98，避免 patch 截断
    transform=train_tf,
)

ds_test = S2CsvDataset(
    csv_path="data_csv/96_3_test.csv",
    ds_cfg_name="s2_12band",
    normalize_stats=None,
    scale_to_unit=True,
    pad_to_multiple=14,
    transform=None,
)

train_loader = DataLoader(ds_train, batch_size=64, shuffle=True, num_workers=8)
test_loader = DataLoader(ds_test, batch_size=64, shuffle=False, num_workers=8)


In [9]:
from dinov2.data.datasets.s2_csv import S2CsvDataset

csv = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s2_90360_temporal_CDSE0_gee90360_2024_16/train.csv"
subset = 1000  # 2% 抽样，可改为 int 比如 500
ds = S2CsvDataset(
    csv_path=csv,
    ds_cfg_name="s2_12band",
    scale_to_unit=False,
    compute_stats=True,
    compute_stats_subset=subset,
    pad_to_multiple=None,  # 统计时不需要 padding
)

mean, std = ds.get_normalize_stats()
print("mean =", mean.tolist())
print("std  =", std.tolist())

csv = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s2_90360_temporal_CDSE0_gee90360_2024_16/test.csv"
subset = 1000  # 2% 抽样，可改为 int 比如 500
ds = S2CsvDataset(
    csv_path=csv,
    ds_cfg_name="s2_12band",
    scale_to_unit=False,
    compute_stats=True,
    compute_stats_subset=subset,
    pad_to_multiple=None,  # 统计时不需要 padding
)

mean, std = ds.get_normalize_stats()
print("mean =", mean.tolist())
print("std  =", std.tolist())


mean = [1834.1697998046875, 1997.9698486328125, 2386.554443359375, 2793.37060546875, 3182.25830078125, 3513.309814453125, 3685.052001953125, 3930.157958984375, 0.0, 0.0, 4635.279296875, 3836.010009765625]
std  = [328.0159912109375, 454.60638427734375, 554.505615234375, 723.8554077148438, 740.930419921875, 683.3168334960938, 673.1165771484375, 652.6875, 9.999999974752427e-07, 9.999999974752427e-07, 679.613525390625, 731.38232421875]
mean = [2201.8984375, 2484.21337890625, 2947.541748046875, 3823.814208984375, 4148.443359375, 4329.40576171875, 4488.68310546875, 4666.5751953125, 0.0, 0.0, 5810.994140625, 5510.0078125]
std  = [484.7310485839844, 645.2403564453125, 760.5748901367188, 910.9255981445312, 951.3125, 925.4602661132812, 925.116943359375, 906.8043212890625, 9.999999974752427e-07, 9.999999974752427e-07, 681.802978515625, 640.1221923828125]


In [10]:
import pandas as pd
import tempfile
from pathlib import Path

from dinov2.data.datasets.s2_csv import S2CsvDataset

# ======================
# Config
# ======================
TRAIN_CSV = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s2_90360_temporal_CDSE0_gee90360_2024_16/train.csv"
TEST_CSV  = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s2_90360_temporal_CDSE0_gee90360_2024_16/test.csv"

N_TRAIN = 1000
N_TEST  = 1000
RANDOM_SEED = 42

# ======================
# 1) Load & sample
# ======================
df_train = pd.read_csv(TRAIN_CSV)
df_test  = pd.read_csv(TEST_CSV)

df_train_sub = df_train.sample(n=N_TRAIN, random_state=RANDOM_SEED)
df_test_sub  = df_test.sample(n=N_TEST,  random_state=RANDOM_SEED)

df_mix = pd.concat([df_train_sub, df_test_sub], ignore_index=True)

print(f"[INFO] Mixed samples: train={len(df_train_sub)}, test={len(df_test_sub)}, total={len(df_mix)}")

# ======================
# 2) Write temp CSV
# ======================
tmp_dir = Path(tempfile.mkdtemp())
MIXED_CSV = tmp_dir / "train_test_mixed_2000.csv"
df_mix.to_csv(MIXED_CSV, index=False)

print(f"[INFO] Mixed CSV saved to: {MIXED_CSV}")

# ======================
# 3) Compute stats
# ======================
ds = S2CsvDataset(
    csv_path=str(MIXED_CSV),
    ds_cfg_name="s2_12band",
    scale_to_unit=False,
    compute_stats=True,
    compute_stats_subset=None,  # 已经是 2000，全用
    pad_to_multiple=None,
)

mean, std = ds.get_normalize_stats()

print("mean =", mean.tolist())
print("std  =", std.tolist())


[INFO] Mixed samples: train=1000, test=1000, total=2000
[INFO] Mixed CSV saved to: /tmp/tmphmho_tkd/train_test_mixed_2000.csv
mean = [2023.93408203125, 2267.178466796875, 2703.1396484375, 3257.457275390625, 3589.15478515625, 3874.7529296875, 4058.203125, 4229.97802734375, 0.0, 0.0, 4915.81591796875, 4429.00439453125]
std  = [659.8005981445312, 759.5709838867188, 878.8653564453125, 1164.0340576171875, 1186.4278564453125, 1077.7305908203125, 1084.9752197265625, 1064.8203125, 9.999999974752427e-07, 9.999999974752427e-07, 1363.797119140625, 1409.4805908203125]


In [6]:
import os
import pandas as pd
import tifffile
from collections import Counter

csv = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_s2_-790360_32_2024/train_patches.csv"
df = pd.read_csv(csv)

cnt = Counter()
bad = []
for p in df["s2_pre_pre_path"].dropna().astype(str):
    if not os.path.exists(p):
        cnt["missing"] += 1
        continue
    try:
        arr = tifffile.imread(p)
        cnt[str(arr.shape)] += 1
        # 记录一下异常的
        if not (arr.ndim == 3 and arr.shape[0] == 12 and arr.shape[1] == 32 and arr.shape[2] == 32):
            bad.append((p, arr.shape))
    except Exception as e:
        cnt["read_fail"] += 1
        bad.append((p, f"FAIL:{e}"))

print("Shape counts:")
for k,v in cnt.most_common():
    print(v, k)

print("\nFirst 20 abnormal samples:")
for item in bad[:20]:
    print(item)


Shape counts:
4288 (12, 32, 32)

First 20 abnormal samples:


In [ ]:
import numpy as np
import tifffile

def plume_ratio(mask_path):
    m = tifffile.imread(mask_path)
    return (m > 0).mean()

# 分别算 train/test 的 label=1
mask_path="/home/yuyao/panopticon/data_csv/96_3_train.csv"
print(plume_ratio(mask_path))

In [6]:
import pandas as pd
from pathlib import Path

csv_path = Path("data_csv/hongxuan_temporal_32/test.csv")

df = pd.read_csv(csv_path)
df.to_csv(csv_path.with_suffix(".bak"), index=False)

df["plume_mask_path"] = df["plume_mask_path"].str.replace(
    r"^/home", "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao", regex=True
)

df.to_csv(csv_path, index=False)

In [3]:
from dinov2.data.datasets.landsat_csv import Landsat89CsvDataset

train_csv = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_l89_32_fixed_512/train.csv"
ds = Landsat89CsvDataset(
    csv_path=train_csv,
    ds_cfg_name="landsat89_7band",
    scale_to_unit=False,        # 若已是反射率则设 False
    compute_stats=True,
    compute_stats_subset=500,  # 可选，加速
    pad_to_multiple=None,
    drop_extra_bands=True,     # 只保留 SR_B1..SR_B7
)
mean, std = ds.get_normalize_stats()
print("mean =", [float(m) for m in mean])
print("std  =", [float(s) for s in std])

/home/yuyao/panopticon/dinov2/data/datasets/s2_csv.py:154: UserWarning: Input has 10 channels; keeping first 7 (dropping QA bands).
  img = self._read_image_raw(path).double()


mean = [10928.7470703125, 11603.849609375, 13416.6435546875, 15134.5927734375, 18313.583984375, 20491.015625, 18536.9921875]
std  = [1211.1807861328125, 1373.5697021484375, 1758.531005859375, 2193.44873046875, 2238.14990234375, 2352.88671875, 2125.662353515625]


In [4]:
import numpy as np
import pandas as pd
from pathlib import Path

CSV_PATH = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s5p_patches_3x3_to_32_offl/train_samples.csv"
NPZ_KEY = None          # 如果 NPZ 里有特定 key，填字符串；否则用第一个数组
ALLOW_PICKLE = True     # 如果保存方式需要 pickle，设为 True
NAN_TO_NUM = 0.0        # 将 NaN/Inf 替换成该值；如不需要可设为 None

df = pd.read_csv(CSV_PATH)
paths = df["image_path"].tolist()

sum_c = None
sumsq_c = None
count = 0

for i, p in enumerate(paths, 1):
    with np.load(p, allow_pickle=ALLOW_PICKLE) as npz:
        arr = np.array(npz[NPZ_KEY]) if NPZ_KEY is not None else np.array(npz[npz.files[0]])
    # 确保形状为 (C,H,W)
    if arr.ndim == 2:
        arr = arr[None, ...]
    elif arr.ndim == 3 and arr.shape[0] not in (1, 3):
        arr = np.transpose(arr, (2, 0, 1))
    if NAN_TO_NUM is not None:
        arr = np.nan_to_num(arr, nan=NAN_TO_NUM, posinf=NAN_TO_NUM, neginf=NAN_TO_NUM)

    if sum_c is None:
        c = arr.shape[0]
        sum_c = np.zeros(c, dtype=np.float64)
        sumsq_c = np.zeros(c, dtype=np.float64)
    count += arr.shape[1] * arr.shape[2]
    sum_c += arr.sum(axis=(1, 2))
    sumsq_c += (arr * arr).sum(axis=(1, 2))

    if i % 500 == 0:
        print(f"Processed {i}/{len(paths)} files...")

mean = sum_c / count
std = np.sqrt((sumsq_c / count) - mean**2)

print("mean:", mean.tolist())
print("std:", std.tolist())

Processed 500/6222 files...
Processed 1000/6222 files...
Processed 1500/6222 files...
Processed 2000/6222 files...
Processed 2500/6222 files...
Processed 3000/6222 files...
Processed 3500/6222 files...
Processed 4000/6222 files...
Processed 4500/6222 files...
Processed 5000/6222 files...
Processed 5500/6222 files...
Processed 6000/6222 files...
mean: [1908.944798144908, 362.20128748570943, 666.5645739544017]
std: [52.2615669211965, 743.7553740768346, 901.8399543163595]


In [ ]:
# 把 csv 路径和列名改成你自己的
csv='/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/data_dir_l89_L2SR/l89_temporal_32_resized_to_224/train.csv'
cols="path_t0 path_t90 path_t360"

import pandas as pd, tifffile as tiff
from collections import Counter
cols = cols.split()
df = pd.read_csv(csv)
hist = Counter()

for i, row in df.head(5).iterrows():  # 先检查前 5 条，可改大
    print(f"--- sample {i} ---")
    for c in cols:
        path = row[c]
        arr = tiff.imread(path)
        hist[arr.shape[0]] += 1
        print(f"{c}: shape={arr.shape}, dtype={arr.dtype}, path={path}")
print("channel count histogram:", hist)



--- sample 0 ---


KeyError: 'p'